# California Housing Price Prediction using Linear Regression

**Internship Project - Artificial Intelligence & Machine Learning Task 1**

This notebook builds and evaluates a **Linear Regression House Price Predictor** using the **California Housing Dataset** available in `scikit-learn` through `fetch_california_housing()`.

## Project Deliverables

- Complete Jupyter Notebook with code, comments, graphs, and conclusions
- Short PDF report generated from the notebook outputs
- Saved machine learning model using Pickle/Joblib
- GitHub-ready project structure with README and requirements

## Objective

The objective of this project is to understand the complete machine learning workflow: data loading, exploration, preprocessing, model training, evaluation, reporting, and saving the model for future use.

## 1. Import Required Libraries

We import libraries for data handling, visualization, model training, evaluation, and saving project outputs.

In [ ]:
# ===============================
# 1. IMPORT REQUIRED LIBRARIES
# ===============================

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning dataset and model
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Model evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Saving model and files
import joblib
from pathlib import Path

# Ignore warnings for clean output
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

# Create folders for project outputs
Path('outputs').mkdir(exist_ok=True)
Path('outputs/plots').mkdir(exist_ok=True)
Path('outputs/model').mkdir(exist_ok=True)
Path('outputs/reports').mkdir(exist_ok=True)

print('Libraries imported successfully!')

## 2. Load the California Housing Dataset

The California Housing Dataset contains **20,640 samples**, **8 numerical input features**, and one target variable representing median house value in units of 100,000 dollars.

In [ ]:
# ===============================
# 2. LOAD DATASET
# ===============================

# Load dataset as a pandas DataFrame
housing = fetch_california_housing(as_frame=True)

# Feature data
X_raw = housing.data

# Target data
# MedHouseVal = Median house value for California districts in units of $100,000
y_raw = housing.target

# Combine features and target into one DataFrame for analysis
df = X_raw.copy()
df['MedHouseVal'] = y_raw

print('Dataset loaded successfully!')
print('Dataset shape:', df.shape)
df.head()

## 3. Dataset Feature Description

| Feature | Meaning |
|---|---|
| MedInc | Median income in block group |
| HouseAge | Median house age in block group |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block group population |
| AveOccup | Average number of household members |
| Latitude | Geographic latitude |
| Longitude | Geographic longitude |
| MedHouseVal | Target variable: median house value in units of $100,000 |

## 4. Basic Data Inspection

This step checks dataset structure, datatypes, summary statistics, and missing values.

In [ ]:
# ===============================
# 4. BASIC DATA INSPECTION
# ===============================

print('First 5 rows:')
display(df.head())

print('
Dataset information:')
df.info()

print('
Statistical summary:')
display(df.describe().T)

In [ ]:
# Check missing values
missing_values = df.isnull().sum().to_frame(name='Missing Values')
missing_values['Percentage'] = (missing_values['Missing Values'] / len(df)) * 100
missing_values

### Observation

The dataset is already clean and numerical. Since there are no missing values, no imputation is required. This makes it suitable for a beginner-friendly regression workflow.

## 5. Exploratory Data Analysis (EDA)

EDA helps us understand data distribution, outliers, and relationships between features and the target variable.

In [ ]:
# ===============================
# 5. TARGET VARIABLE DISTRIBUTION
# ===============================

plt.figure(figsize=(10, 6))
sns.histplot(df['MedHouseVal'], bins=50, kde=True)
plt.title('Distribution of Median House Value', fontsize=14)
plt.xlabel('Median House Value in $100,000 units')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('outputs/plots/target_distribution.png', dpi=300)
plt.show()

In [ ]:
# ===============================
# 6. FEATURE DISTRIBUTIONS
# ===============================

# Plot histograms for all numerical features
feature_columns = housing.feature_names

for col in feature_columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col], bins=40, kde=True)
    plt.title(f'Distribution of {col}', fontsize=13)
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.savefig(f'outputs/plots/distribution_{col}.png', dpi=300)
    plt.show()

In [ ]:
# ===============================
# 7. CORRELATION ANALYSIS
# ===============================

plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=16)
plt.tight_layout()
plt.savefig('outputs/plots/correlation_heatmap.png', dpi=300)
plt.show()

# Show correlation with target variable
correlation_with_target = correlation_matrix['MedHouseVal'].sort_values(ascending=False)
print('Correlation with target variable:')
display(correlation_with_target)

### Key EDA Insight

`MedInc` usually has the strongest positive correlation with house value, meaning areas with higher median income generally have higher house prices. Geographic variables such as `Latitude` and `Longitude` also influence prices because house values vary by location.

In [ ]:
# ===============================
# 8. IMPORTANT FEATURE RELATIONSHIPS
# ===============================

plt.figure(figsize=(9, 6))
sns.scatterplot(x='MedInc', y='MedHouseVal', data=df, alpha=0.4)
plt.title('Median Income vs Median House Value', fontsize=14)
plt.xlabel('Median Income')
plt.ylabel('Median House Value')
plt.tight_layout()
plt.savefig('outputs/plots/medinc_vs_housevalue.png', dpi=300)
plt.show()

plt.figure(figsize=(9, 6))
sns.scatterplot(x='Longitude', y='Latitude', hue='MedHouseVal', data=df, palette='viridis', alpha=0.6)
plt.title('Geographical Distribution of House Values', fontsize=14)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='House Value')
plt.tight_layout()
plt.savefig('outputs/plots/geographical_house_values.png', dpi=300)
plt.show()

## 6. Data Preprocessing

The dataset contains numerical features only. The target variable is separated from the input features before model training.

For standard Linear Regression, feature scaling is not mandatory for prediction performance, but scaling may be useful for models that depend on distance or gradient optimization.

In [ ]:
# ===============================
# 9. DEFINE FEATURES AND TARGET
# ===============================

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)

## 7. Train-Test Split

We split the data into training and testing sets. The model learns from the training data and is evaluated on unseen testing data.

- Training data: 80%
- Testing data: 20%
- Random state: 42 for reproducibility

In [ ]:
# ===============================
# 10. TRAIN-TEST SPLIT
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

## 8. Model Training - Linear Regression

Linear Regression is a supervised machine learning algorithm used to predict continuous numerical values. It assumes a linear relationship between input features and the target variable.

In [ ]:
# ===============================
# 11. TRAIN LINEAR REGRESSION MODEL
# ===============================

model = LinearRegression()
model.fit(X_train, y_train)

print('Linear Regression model trained successfully!')

## 9. Model Prediction

After training, the model is used to predict house values for the test data.

In [ ]:
# ===============================
# 12. MAKE PREDICTIONS
# ===============================

y_pred = model.predict(X_test)

# Show first 10 actual vs predicted values
prediction_comparison = pd.DataFrame({
    'Actual Value': y_test.values[:10],
    'Predicted Value': y_pred[:10]
})

prediction_comparison

## 10. Model Evaluation

We evaluate the model using:

- **MAE**: Average absolute prediction error
- **RMSE**: Penalizes larger errors more strongly
- **R2 Score**: Explains how much variance is captured by the model

In [ ]:
# ===============================
# 13. EVALUATE MODEL
# ===============================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metrics_df = pd.DataFrame({
    'Metric': ['Mean Absolute Error (MAE)', 'Root Mean Squared Error (RMSE)', 'R2 Score'],
    'Value': [mae, rmse, r2]
})

print('Model Evaluation Results:')
display(metrics_df)

# Save metrics
metrics_df.to_csv('outputs/model_evaluation_metrics.csv', index=False)

### Metric Interpretation

A lower MAE and RMSE indicate better prediction accuracy. The R2 Score shows how well the model explains the variation in house prices. A moderate R2 Score is expected because real estate prices are influenced by many complex factors that may not be fully captured by the dataset.

In [ ]:
# ===============================
# 14. ACTUAL VS PREDICTED PLOT
# ===============================

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual House Value')
plt.ylabel('Predicted House Value')
plt.title('Actual vs Predicted House Values')
plt.tight_layout()
plt.savefig('outputs/plots/actual_vs_predicted.png', dpi=300)
plt.show()

In [ ]:
# ===============================
# 15. RESIDUAL ANALYSIS
# ===============================

residuals = y_test - y_pred

plt.figure(figsize=(9, 6))
sns.histplot(residuals, bins=50, kde=True)
plt.title('Residual Distribution')
plt.xlabel('Residual Error')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('outputs/plots/residual_distribution.png', dpi=300)
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Values')
plt.tight_layout()
plt.savefig('outputs/plots/residuals_vs_predicted.png', dpi=300)
plt.show()

## 11. Feature Coefficient Analysis

Linear Regression coefficients show how each feature contributes to the predicted house value when other features are held constant.

In [ ]:
# ===============================
# 16. FEATURE COEFFICIENTS
# ===============================

coefficients_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values(by='Coefficient', ascending=False)

print('Linear Regression Coefficients:')
display(coefficients_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Coefficient', y='Feature', data=coefficients_df)
plt.title('Feature Coefficients in Linear Regression')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('outputs/plots/feature_coefficients.png', dpi=300)
plt.show()

## 12. Save the Trained Model

The trained model is saved using `joblib` so it can be reused without retraining.

In [ ]:
# ===============================
# 17. SAVE TRAINED MODEL
# ===============================

model_path = 'outputs/model/linear_regression_house_price_model.pkl'
joblib.dump(model, model_path)

print(f'Model saved successfully at: {model_path}')

## 13. Prediction Demo on New Input

This section demonstrates how the saved model can be used to make a prediction for a new sample.

In [ ]:
# ===============================
# 18. SAMPLE PREDICTION
# ===============================

# Example input format:
# [MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude]

sample_input = pd.DataFrame([[8.3252, 41.0, 6.9841, 1.0238, 322.0, 2.5556, 37.88, -122.23]],
                            columns=X.columns)

sample_prediction = model.predict(sample_input)

print('Predicted Median House Value:', sample_prediction[0])
print('Approximate value in dollars: $', round(sample_prediction[0] * 100000, 2))

## 14. Generate Project Report PDF Automatically

Run the following cell after model training. It creates a professional PDF report using the calculated model metrics.

In [ ]:
# ===============================
# 19. GENERATE PDF REPORT
# ===============================

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY

report_path = 'outputs/reports/California_Housing_Linear_Regression_Report.pdf'

doc = SimpleDocTemplate(
    report_path,
    pagesize=A4,
    rightMargin=40,
    leftMargin=40,
    topMargin=40,
    bottomMargin=40
)

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name='CenterTitle', parent=styles['Title'], alignment=TA_CENTER, fontSize=18, leading=22))
styles.add(ParagraphStyle(name='Justified', parent=styles['BodyText'], alignment=TA_JUSTIFY, leading=14))

story = []

story.append(Paragraph('California Housing Price Prediction Using Linear Regression', styles['CenterTitle']))
story.append(Spacer(1, 12))
story.append(Paragraph('Internship Project Report - Artificial Intelligence & Machine Learning Task 1', styles['Heading3']))
story.append(Spacer(1, 18))

sections = [
    ('Abstract', 'This project presents a machine learning solution for predicting California housing prices using Linear Regression. The project follows the complete ML workflow including data loading, exploratory data analysis, preprocessing, model training, evaluation, visualization, and model saving.'),
    ('Objective', 'The objective is to build and evaluate a Linear Regression model that predicts median house values using the California Housing Dataset.'),
    ('Dataset Description', 'The dataset contains 20,640 samples, 8 numerical features, and one target variable named MedHouseVal. The target represents median house value in units of 100,000 dollars.'),
    ('Methodology', 'The workflow includes importing libraries, loading the dataset, checking missing values, performing EDA, splitting the data, training the Linear Regression model, evaluating predictions, visualizing results, and saving the trained model.'),
    ('Exploratory Data Analysis', 'EDA showed that Median Income is strongly related to house value. Geographic location represented by latitude and longitude also plays an important role in explaining price variation.'),
    ('Model Development', 'A Linear Regression model was trained using 80% of the data and tested on 20% unseen data. The train-test split was performed with random_state=42 for reproducibility.'),
]

for heading, body in sections:
    story.append(Paragraph(heading, styles['Heading2']))
    story.append(Paragraph(body, styles['Justified']))
    story.append(Spacer(1, 10))

# Metrics table
story.append(Paragraph('Model Evaluation Results', styles['Heading2']))
metrics_table_data = [['Metric', 'Value'], ['MAE', f'{mae:.4f}'], ['RMSE', f'{rmse:.4f}'], ['R2 Score', f'{r2:.4f}']]
metrics_table = Table(metrics_table_data, colWidths=[3*inch, 2*inch])
metrics_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#1f4e79')),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
    ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('BACKGROUND', (0, 1), (-1, -1), colors.whitesmoke),
]))
story.append(metrics_table)
story.append(Spacer(1, 12))

# Add key plots if available
plot_files = [
    ('Correlation Heatmap', 'outputs/plots/correlation_heatmap.png'),
    ('Actual vs Predicted House Values', 'outputs/plots/actual_vs_predicted.png'),
    ('Residual Distribution', 'outputs/plots/residual_distribution.png')
]

for title, path in plot_files:
    if Path(path).exists():
        story.append(Paragraph(title, styles['Heading3']))
        story.append(Image(path, width=5.8*inch, height=3.6*inch))
        story.append(Spacer(1, 10))

story.append(Paragraph('Conclusion', styles['Heading2']))
story.append(Paragraph('The Linear Regression model successfully predicted California housing prices with moderate accuracy. Median Income was found to be one of the most influential features. The project demonstrates a complete beginner-to-intermediate machine learning workflow suitable for internship submission and portfolio presentation.', styles['Justified']))
story.append(Spacer(1, 10))

story.append(Paragraph('Future Scope', styles['Heading2']))
story.append(Paragraph('Future improvements can include feature scaling, advanced feature engineering, Random Forest Regression, Gradient Boosting, XGBoost, hyperparameter tuning, and deployment using Streamlit or Flask.', styles['Justified']))
story.append(Spacer(1, 10))

story.append(Paragraph('References', styles['Heading2']))
story.append(Paragraph('1. Scikit-Learn Documentation - California Housing Dataset<br/>2. Scikit-Learn Documentation - Linear Regression<br/>3. Pandas, NumPy, Matplotlib and Seaborn Documentation', styles['BodyText']))

doc.build(story)

print('PDF report generated successfully:', report_path)

## 15. Create GitHub-Ready Files

This cell creates a professional README and requirements file for GitHub submission.

In [ ]:
# ===============================
# 20. CREATE README AND REQUIREMENTS
# ===============================

readme_content = f"""
# California Housing Price Prediction using Linear Regression

## Project Overview
This project predicts California housing prices using the California Housing Dataset and Linear Regression. It demonstrates the complete machine learning lifecycle: data loading, exploratory data analysis, preprocessing, model training, evaluation, visualization, and model saving.

## Internship Task
Artificial Intelligence & Machine Learning Task 1: Build and evaluate a Linear Regression model for house price prediction.

## Dataset
- Dataset: California Housing Dataset
- Source: Scikit-Learn
- Samples: 20,640
- Features: 8
- Target: Median house value in units of $100,000

## Technologies Used
- Python
- Jupyter Notebook
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-Learn
- Joblib
- ReportLab

## Model Used
- Linear Regression

## Evaluation Metrics
| Metric | Value |
|---|---:|
| MAE | {mae:.4f} |
| RMSE | {rmse:.4f} |
| R2 Score | {r2:.4f} |

## Project Structure
```text
California-Housing-Linear-Regression/
├── task1_ml_linear_regression.ipynb
├── README.md
├── requirements.txt
└── outputs/
    ├── plots/
    ├── model/
    ├── reports/
    └── model_evaluation_metrics.csv
```

## How to Run
1. Clone the repository.
2. Install requirements:
   ```bash
   pip install -r requirements.txt
   ```
3. Open the notebook:
   ```bash
   jupyter notebook task1_ml_linear_regression.ipynb
   ```
4. Run all cells.

## Conclusion
The Linear Regression model provides a baseline prediction system for California housing prices. The model achieved moderate predictive performance and helped identify important features such as Median Income and geographic location.

## Future Improvements
- Try Random Forest Regression
- Try Gradient Boosting or XGBoost
- Perform hyperparameter tuning
- Add feature engineering
- Deploy with Streamlit or Flask
"""

requirements_content = """
pandas
numpy
matplotlib
seaborn
scikit-learn
joblib
reportlab
jupyter
"""

Path('README.md').write_text(readme_content, encoding='utf-8')
Path('requirements.txt').write_text(requirements_content.strip(), encoding='utf-8')

print('README.md and requirements.txt created successfully!')

## Final Conclusion

This internship project successfully demonstrates how to build a complete machine learning regression pipeline using the California Housing Dataset. The project includes data loading, EDA, visualization, model training, prediction, evaluation, model saving, and report generation.

The Linear Regression model works as a strong baseline model. However, since house prices depend on complex non-linear factors, future work can improve performance using advanced algorithms such as Random Forest, Gradient Boosting, and XGBoost.